# Google Search Ranking & Discoverability Capstone

**Track:** Machine Learning  |  **Lane:** Refresh / Content Opportunity Scoring

**Question:** Which content pages should be prioritised for review/refresh based on historical search-performance signals?

> **Important:** The starter CSV is a development/validation dataset. The published capstone result must come from the gated FlyRank daily warehouse because only the warehouse supports genuine historical → future temporal labels. Starter results are never written to the public paper.


## §1 — Project Setup

In [ ]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    pass

# Use "starter" to validate the code locally without credentials.
# Use "warehouse" for the actual Week-8 capstone experiment and paper.
DATA_MODE = "starter"
CUTOFFS = ["2026-03-01", "2026-04-01", "2026-05-01", "2026-06-01"]
HIST_DAYS = 90
FUTURE_DAYS = 30
MIN_HIST_IMPRESSIONS = 50
OUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUT_FIGS = PROJECT_ROOT / "outputs" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True); OUT_FIGS.mkdir(parents=True, exist_ok=True)
print(f"ROOT={PROJECT_ROOT} | MODE={DATA_MODE}")


## §2 — Data Loading & Contract

In [ ]:
from src import data as DM

if DATA_MODE == "starter":
    raw_df = DM.load_starter_csv()
    print(f"Starter CSV: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
    print(f"Clients: {raw_df.client_id.nunique():,} | Pages: {raw_df.content_id.nunique():,}")
elif DATA_MODE == "warehouse":
    from src import warehouse as WH
    con = DM.get_warehouse_connection()
    schema = WH.discover_schema(con)
    print("Warehouse schema:", schema)
else:
    raise ValueError("DATA_MODE must be 'starter' or 'warehouse'")


## §3 — Starter Data: Development Validation Only

In [ ]:
if DATA_MODE == "starter":
    prepared_df = DM.prepare_starter_features(raw_df)
    X_full, y_full = DM.starter_feature_matrix(prepared_df)
    DM.assert_no_label_leakage_starter(X_full)
    print("Prepared pages:", len(prepared_df))
    print("Starter label distribution:", y_full.value_counts().to_dict())
    print("NOTE: this is a cross-sectional snapshot; it is NOT the final temporal capstone experiment.")


## §4 — Warehouse Temporal Snapshots (Final Experiment)

In [ ]:
if DATA_MODE == "warehouse":
    from src import warehouse as WH
    snapshots = []
    for cutoff in CUTOFFS:
        print(f"Building snapshot: cutoff={cutoff}, history={HIST_DAYS}d, future={FUTURE_DAYS}d")
        snap = WH.build_temporal_snapshot(con, schema, cutoff, HIST_DAYS, FUTURE_DAYS, MIN_HIST_IMPRESSIONS)
        if snap.empty:
            print("  -> no eligible rows")
            continue
        snap["cutoff"] = pd.to_datetime(cutoff)
        snapshots.append(snap.reset_index(names="page_id"))
        print(f"  -> {len(snap):,} eligible page snapshots")
    if not snapshots:
        raise RuntimeError("No temporal snapshots were produced. Check CUTOFFS, warehouse URI/schema, and access.")
    temporal = pd.concat(snapshots, ignore_index=True)
    temporal = temporal.dropna(subset=["label"]).copy()
    temporal["label"] = temporal["label"].astype(int)
    print("Temporal dataset:", temporal.shape)
    print("Cutoffs:", temporal["cutoff"].dt.strftime("%Y-%m-%d").value_counts().sort_index().to_dict())


## §5 — Leakage Audit

In [ ]:
from src import evaluation as EM
audit = EM.leakage_audit_table()
display(audit)
audit.to_csv(OUT_TABLES / "leakage_audit.csv", index=False)
if DATA_MODE == "warehouse":
    feature_cols = [c for c in temporal.columns if c not in {"page_id","cutoff","hist_start","future_end","label","relative_change"}]
    forbidden = {"label","relative_change","future_rate","future_clicks","future_impressions"}
    leaked = sorted(set(feature_cols) & forbidden)
    assert not leaked, f"Future/label columns leaked into features: {leaked}"
    print("[leakage] Temporal feature columns contain no future/label fields ✓")


## §6 — Exploratory Checks

In [ ]:
if DATA_MODE == "starter":
    vc = prepared_df["trend_direction"].value_counts()
    print("Starter trend distribution:", vc.to_dict())
else:
    print(temporal.groupby("cutoff")["label"].agg(["count","mean"]).round(3))


## §7 — Train / Validation / Test

In [ ]:
if DATA_MODE == "starter":
    train_df, val_df, test_df = DM.three_way_client_split(prepared_df, val_fraction=0.15, test_fraction=0.15, seed=42)
    X_train, y_train = DM.starter_feature_matrix(train_df)
    X_val, y_val = DM.starter_feature_matrix(val_df)
    X_test, y_test = DM.starter_feature_matrix(test_df)
    X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    print("STARTER DEVELOPMENT SPLIT — client holdout, not final temporal evaluation")
else:
    temporal = temporal.sort_values(["cutoff", "page_id"]).reset_index(drop=True)
    unique_cutoffs = sorted(temporal["cutoff"].unique())
    if len(unique_cutoffs) < 3:
        raise RuntimeError("Need at least 3 distinct cutoff dates for train/validation/test.")
    n_train = max(1, len(unique_cutoffs)-2)
    train_cutoffs = unique_cutoffs[:n_train]
    val_cutoff = unique_cutoffs[n_train]
    test_cutoff = unique_cutoffs[n_train+1]
    train_df = temporal[temporal.cutoff.isin(train_cutoffs)].copy()
    val_df = temporal[temporal.cutoff == val_cutoff].copy()
    test_df = temporal[temporal.cutoff == test_cutoff].copy()
    feature_cols = [c for c in temporal.columns if c not in {"page_id","cutoff","hist_start","future_end","label","relative_change"}]
    X_train, y_train = train_df[feature_cols].astype(float), train_df["label"]
    X_val, y_val = val_df[feature_cols].astype(float), val_df["label"]
    X_test, y_test = test_df[feature_cols].astype(float), test_df["label"]
    X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    assert train_df.cutoff.max() < val_df.cutoff.min() < test_df.cutoff.min()
    print("TEMPORAL SPLIT")
    print("train cutoffs:", [str(x)[:10] for x in train_cutoffs])
    print("validation cutoff:", str(val_cutoff)[:10])
    print("test cutoff:", str(test_cutoff)[:10])
print(f"Train={X_train.shape} | Val={X_val.shape} | Test={X_test.shape}")


## §8 — Baseline

In [ ]:
from src import models as MM
if DATA_MODE == "starter":
    baseline_val = MM.baseline_decline_score_starter(X_val)
    baseline_test = MM.baseline_decline_score_starter(X_test)
else:
    baseline_val = MM.baseline_decline_score_temporal(val_df.set_index("page_id")[MM.baseline_decline_score_temporal.__annotations__.keys()] if False else val_df.set_index("page_id"))
    baseline_test = MM.baseline_decline_score_temporal(test_df.set_index("page_id"))
print("Validation baseline:", EM.classification_metrics(y_val.to_numpy(), baseline_val.to_numpy()))


## §9 — Candidate Models

In [ ]:
candidates = MM.train_all_candidates(X_train, y_train)
val_results = EM.evaluate_all_candidates(candidates, X_val, y_val)
display(val_results.round(4))


## §10 — Validation-Based Model Selection

In [ ]:
val_results_list = val_results.to_dict(orient="records")
best_model_name = EM.select_best_model(val_results_list, primary_metric="pr_auc")
best_model = next(m for m in candidates if m.name == best_model_name)
print("Selected on validation only:", best_model_name)


## §11 — Final Held-Out Test Evaluation

In [ ]:
test_scores_model = best_model.predict_proba(X_test)
test_scores_baseline = baseline_test.to_numpy()
model_test_m = EM.classification_metrics(y_test.to_numpy(), test_scores_model)
baseline_test_m = EM.classification_metrics(y_test.to_numpy(), test_scores_baseline)
comparison = EM.compare_baseline_vs_model(y_test.to_numpy(), test_scores_baseline, test_scores_model, [10,25,50,100])
print("MODEL TEST:", model_test_m)
print("BASELINE TEST:", baseline_test_m)
display(comparison.round(4))


## §12 — Ranked Recommendations

In [ ]:
from src import recommendations as RM
if DATA_MODE == "starter":
    X_all, _ = DM.starter_feature_matrix(prepared_df)
    X_all_aligned = X_all.reindex(columns=X_train.columns, fill_value=0)
    all_scores = pd.Series(best_model.predict_proba(X_all_aligned), index=X_all_aligned.index, name="score")
    feat = __import__("src.features", fromlist=["build_starter_features"]).build_starter_features(prepared_df)
    recs = RM.generate_recommendations(feat, all_scores, top_n=100, mode="starter")
    print("Starter development recommendations generated. These are NOT published as final capstone results.")
else:
    latest = test_df.copy().set_index("page_id")
    scores = pd.Series(test_scores_model, index=latest.index, name="score")
    rec_features = latest[[c for c in ["click_trend","impression_trend","position_trend","click_volatility","impressions_hist"] if c in latest.columns]]
    recs = RM.generate_recommendations(rec_features, scores, top_n=100, mode="temporal")
    print("Final test-period recommendations:")
    display(recs.head(20))
recs.to_csv(OUT_TABLES / ("starter_development_recommendations.csv" if DATA_MODE=="starter" else "ranked_recommendations.csv"), index=False)


## §13 — Export Results

In [ ]:
if DATA_MODE == "starter":
    print("Starter pipeline completed. No public experiment_results.json is written from the starter snapshot.")
else:
    results_summary = {
        "data_mode": "warehouse",
        "n_page_snapshots": int(len(temporal)),
        "n_train": int(len(X_train)), "n_val": int(len(X_val)), "n_test": int(len(X_test)),
        "train_cutoffs": [str(x)[:10] for x in train_cutoffs],
        "validation_cutoff": str(val_cutoff)[:10], "test_cutoff": str(test_cutoff)[:10],
        "best_model": best_model_name,
        "validation_metrics": {m["model"]:{k:round(v,4) for k,v in m.items() if k!="model"} for m in val_results_list},
        "test_metrics_model": {k:(round(v,4) if isinstance(v,float) else v) for k,v in model_test_m.items()},
        "test_metrics_baseline": {k:(round(v,4) if isinstance(v,float) else v) for k,v in baseline_test_m.items()},
        "n_recommendations": int(len(recs)),
        "hist_days": HIST_DAYS, "future_days": FUTURE_DAYS, "decline_threshold": 0.30,
    }
    with open(OUT_TABLES / "experiment_results.json", "w") as f: json.dump(results_summary, f, indent=2)
    comparison.to_csv(OUT_TABLES / "test_baseline_vs_model.csv", index=False)
    EM.confusion_matrix_table(y_test.to_numpy(), test_scores_model).to_csv(OUT_TABLES / "test_confusion_matrix.csv")
    print("FINAL WAREHOUSE RESULTS EXPORTED")


## §14 — Paper Update

In [ ]:
if DATA_MODE == "warehouse":
    import subprocess
    r = subprocess.run([sys.executable, str(PROJECT_ROOT/"scripts"/"update_paper.py")], capture_output=True, text=True, cwd=str(PROJECT_ROOT))
    print(r.stdout)
    if r.returncode != 0: print(r.stderr)
else:
    print("Paper update intentionally skipped in starter mode. Run the notebook with DATA_MODE='warehouse' after authorized FlyRank access is configured.")


## §15 — Final Capstone Checklist

In [ ]:
print("Final checks:")
print("- Real warehouse experiment:", "READY" if DATA_MODE=="warehouse" else "NOT RUN — switch DATA_MODE to 'warehouse'")
print("- Starter pipeline:", "available for development" if DATA_MODE=="starter" else "available")
print("- Public paper:", "updated only from warehouse results")
print("- Credentials:", "must remain in .env / environment, never committed")
